# Double Zernike Analysis (v1)

**Author:** Aaron Roodman  
**Date Created:** 2026-04-12  
**Last Modified:** 2026-04-12  
**Status:** Draft  
**Keywords:** Double Zernike, AOS, intrinsics, correlations, LUT, rotator, elevation, thermal  

## Description

Aggregate Double Zernike (DZ) fit results across all pipeline chunks (OCS or CCS
coordinate system) and run cross-coefficient and operational analyses.

Key functionality:
1. Load and concatenate all `*_fits.parquet` (OCS) or `*_ccs_fits.parquet` (CCS)
   from the pipeline `output/` tree.
2. Reproduce DZ-vs-thermal scatter plots from `intrinsics_thermal_correlations`.
3. Full pairwise correlation analysis over all DZ coefficients; pull out the
   10 largest off-diagonal correlations and make scatter plots.
4. LUT exploration: DZ terms vs rotator angle (elevation ∈ 70 ± 3 deg) and
   DZ terms vs elevation (rotator angle ∈ 0 ± 3 deg).

**Output:** PDF in `output/dz_analysis_{coord_sys}.pdf`  
**Based on:** intrinsics_thermal_correlations.ipynb, run_pipeline outputs

## Change Log

| Date | Author | Description |
|------|--------|-------------|
| 2026-04-12 | Aaron Roodman | Initial version |
| 2026-04-14 | Aaron Roodman | Add thermal-by-variable grid; split large plot sets into separate PDFs; add k1-k2 pairwise scan by (j1,j2); unify radial-trio color scale |
| 2026-04-19 | Aaron Roodman | Extend thermal grid and pairwise scan to j=4..17; add DZ × thermal correlation-matrix heatmap (§6.2) |

## Table of Contents

1. [Parameters](#params)
2. [Setup & Imports](#setup)
3. [Helper Functions](#functions)
4. [Data Access](#data)
5. [Thermal Correlations — per-DZ-term](#thermal-per-term)
6. [Thermal Correlations — overview](#thermal-by-var)
    - [6.1 Per-variable grid](#thermal-by-var-grid)
    - [6.2 DZ × thermal correlation matrix](#thermal-by-var-matrix)
7. [DZ Correlation Matrix](#corr-matrix-sec)
8. [Large DZ Correlations Grouped by Pupil Pair](#corr-groups)
9. [Expected Astigmatism-Symmetry Correlations](#corr-expected)
10. [Pairwise (k1,j1) vs (k2,j2) scan](#pairwise)
11. [LUT: DZ vs Rotator and Elevation](#lut)
12. [Cylindrical (Radial) Pupil Zernike Combinations](#radial)
    - [12.1 Radial DZ Correlation Matrix](#radial-corr)
    - [12.2 Radial Trio Plots](#radial-trio)
13. [Output PDFs](#output)

<a id='params'></a>
## Parameters

In [ ]:
# ============================================================
# Parameters — All configurable values collected here
# ============================================================

# Coordinate system for fit tables: 'OCS' or 'CCS'
coord_sys = 'OCS'

# Where to look for fit parquet files (relative to this notebook)
output_root = 'output'

# Glob patterns: OCS uses *_fits.parquet, CCS uses *_ccs_fits.parquet
fit_glob = {
    'OCS': '*[!_ccs]_fits.parquet',
    'CCS': '*_ccs_fits.parquet',
}[coord_sys]

# ----- Chunk selection -----
# Pick ONE mode (leave the others None):
#   chunk_names       — explicit list (or single string) of run names
#                       from runs.yaml, e.g.
#                         chunk_names = 'chunk5'
#                         chunk_names = ['chunk1', 'chunk2', 'chunk3']
#   param_set_filter  — single param_set name or substring, matches any
#                       chunk whose param_set contains it,
#                       e.g. 'fam_danish_v1_triplets'
#   (both None)       — load every *_fits.parquet found under output_root
chunk_names = None
param_set_filter = None

# DZ fit prefix to use for analysis (z1toz3 or z1toz6)
dz_prefix = 'z1toz6'

# Quality cuts
max_coeff_um = 2.0  # reject visits with any DZ coefficient > this (µm)

# Thermal variables (only used if present in the merged tables)
thermal_vars = [
    'cam_air_temp', 'm2_air_temp', 'm1m3_air_temp', 'outside_temp',
    'm2_delta_t', 'cam_m1m3_delta_t', 'dome_delta_t',
    'x_gradient', 'y_gradient', 'z_gradient', 'radial_gradient',
    'tma_truss_temp_pxpy', 'tma_truss_temp_mxmy',
]

# DZ terms to plot vs thermal (k = focal Noll, j = pupil Noll).
# Include all focal k for pupil j = 4 (defocus) plus astig/trefoil combos.
dz_terms_for_thermal = [
    (1, 4), (2, 4), (3, 4), (4, 4), (5, 4), (6, 4),  # all k, pupil defocus
    (5, 5), (6, 6),                                    # astig-astig
    (5, 10), (6, 9),                                   # astig-trefoil
]

# Per-variable thermal grid: focal k range and pupil j range
# j runs up through 17 so both 2nd Coma (Z16, Z17) are included.
thermal_grid_k_range = range(1, 7)      # k = 1..6 (6 values)
thermal_grid_j_range = range(4, 18)     # j = 4..17 (14 values)

# Pairwise (k1,j1) vs (k2,j2) scan: one page per (j1, j2) pair
# j extended to 17 so 2nd Coma Zernike terms are captured.
pairwise_j_range = range(4, 18)         # j = 4..17 (14 values -> 196 pages)
pairwise_k_range = range(1, 7)          # k = 1..6 (36 subplots per page)

# Correlation scatter plots: minimum |r| to include
corr_threshold = 0.6

# Expected astigmatism-symmetry correlations in the (k, j) basis.
expected_astig_pairs = [
    ((5, 5), (6, 6)),
    ((6, 5), (5, 6)),
    ((1, 5), (1, 6)),
    ((4, 5), (4, 6)),
    ((2, 5), (3, 6)),
    ((3, 5), (2, 6)),
]

# LUT slice tolerances (all in degrees)
elev_center_for_rotator_scan = 70.0
elev_tol = 3.0
rot_center_for_elev_scan = 0.0
rot_tol = 3.0

# Per-run output subdirectory under output_root.  If None, a label is
# auto-generated after QC from coord_sys, the chunk filter, and the
# loaded day_obs range.  Set explicitly to override, e.g.
#     output_label = 'dz_analysis_OCS_danish_v1_mar-apr'
output_label = None

<a id='setup'></a>
## Setup & Imports

In [ ]:
import os
import re
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

sys.path.insert(0, str(Path.cwd().parent))
from common.utils import setup_plotting

setup_plotting()

<a id='functions'></a>
## Helper Functions

In [ ]:
def find_fit_files(output_root, coord_sys,
                   chunk_names=None, param_set_filter=None):
    """Locate DZ fit parquet files for the requested coord system.

    Selection modes (at most one should be set):
      * chunk_names — list of run names from runs.yaml; each is resolved
        via run_pipeline.fits_path / fits_path_ccs.
      * param_set_filter — string; any chunk whose param_set *contains*
        this string is selected.
      * None — glob output_root for every matching fit file (default).

    OCS files end '_fits.parquet' (and NOT '_ccs_fits.parquet').
    CCS files end '_ccs_fits.parquet'.
    """
    is_ccs = coord_sys == 'CCS'

    def _only_coord(paths):
        if is_ccs:
            return [p for p in paths if str(p).endswith('_ccs_fits.parquet')]
        return [p for p in paths
                if str(p).endswith('_fits.parquet')
                and not str(p).endswith('_ccs_fits.parquet')]

    # Accept a bare string for chunk_names as a convenience
    if isinstance(chunk_names, str):
        chunk_names = [chunk_names]

    if chunk_names or param_set_filter:
        # Resolve specific chunks from runs.yaml
        sys.path.insert(0, 'code')
        from run_pipeline import (
            load_runs as _load_runs,
            load_param_sets as _load_param_sets,
            resolve_run as _resolve_run,
            fits_path as _fits_path,
            fits_path_ccs as _fits_path_ccs,
        )
        data = _load_runs()
        param_sets = _load_param_sets()
        runs = data.get('runs', {})

        if chunk_names:
            selected = [(n, runs[n]) for n in chunk_names if n in runs]
            missing = [n for n in chunk_names if n not in runs]
            if missing:
                raise ValueError(f'Unknown chunk(s) in runs.yaml: {missing}')
        else:
            selected = [(n, cfg) for n, cfg in runs.items()
                        if param_set_filter in (cfg.get('param_set') or '')]
            if not selected:
                raise ValueError(
                    f'No runs in runs.yaml match param_set_filter='
                    f'{param_set_filter!r}')

        paths = []
        for name, cfg in selected:
            resolved = _resolve_run(cfg, param_sets)
            fp = _fits_path_ccs(resolved) if is_ccs else _fits_path(resolved)
            # run_pipeline returns 'output/...'; treat as repo-relative
            fp_path = Path(fp)
            if not fp_path.is_absolute():
                fp_path = Path(output_root).parent / fp_path if \
                    Path(output_root).name != 'output' else fp_path
            # Normal case: run_pipeline's fits_path already uses output/...
            paths.append(Path(fp))
        paths = _only_coord(paths)
        # Only return files that actually exist
        return [p for p in paths if Path(p).exists()]

    # Fall-through: glob every fit file in output_root
    root = Path(output_root)
    all_fits = sorted(root.glob('*_fits.parquet'))
    return _only_coord(all_fits)


def load_all_fits(files, source_label='source'):
    """Load and concatenate fit parquet tables, tagging each row with its source."""
    frames = []
    for p in files:
        df = pd.read_parquet(p)
        df[source_label] = Path(p).stem
        frames.append(df)
        print(f"  {Path(p).name}: {len(df)} rows, {len(df.columns)} cols")
    if not frames:
        raise FileNotFoundError('No fit parquet files found')
    combined = pd.concat(frames, ignore_index=True, sort=False)
    return combined


def dz_coeff_columns(df, prefix):
    """Return all DZ coefficient columns for a given prefix.

    Columns are named '{prefix}_z{j}_c{k}' where j = pupil Noll, k = focal Noll.
    Excludes '_err', '_scale', '_bad_fit'.
    """
    pattern = re.compile(rf'^{re.escape(prefix)}_z\d+_c\d+$')
    return [c for c in df.columns if pattern.match(c)]


def parse_jk(col, prefix):
    """Parse '(j, k)' from a column like 'z1toz6_z9_c5'."""
    m = re.match(rf'^{re.escape(prefix)}_z(\d+)_c(\d+)$', col)
    if not m:
        return None
    return int(m.group(1)), int(m.group(2))


# Noll-index to human-readable name mappings
FOCAL_NAMES = {
    1: 'Piston', 2: 'Tilt', 3: 'Tip',
    4: 'Focus', 5: 'Astig45', 6: 'Astig0',
}
PUPIL_NAMES = {
    4: 'Defocus', 5: 'Astig45', 6: 'Astig0',
    7: 'Coma_y', 8: 'Coma_x', 9: 'Trefoil_y', 10: 'Trefoil_x',
    11: 'Spherical', 12: '2ndAstig0', 13: '2ndAstig45',
    14: 'Tetrafoil_x', 15: 'Tetrafoil_y',
    16: '2ndComa_x', 17: '2ndComa_y', 18: '2ndTrefoil_x', 19: '2ndTrefoil_y',
    20: 'Pentafoil_x', 21: 'Pentafoil_y',
    22: '2ndSpherical', 23: '3rdAstig45', 24: '3rdAstig0',
}


def dz_label(col, prefix):
    """Descriptive label like 'Focus of Coma_x (k=4, j=7)'."""
    jk = parse_jk(col, prefix)
    if jk is None:
        return col
    j, k = jk
    focal = FOCAL_NAMES.get(k, f'k{k}')
    pupil = PUPIL_NAMES.get(j, f'Z{j}')
    return f'{focal} of {pupil} (k={k}, j={j})'


def short_label(col, prefix):
    j, k = parse_jk(col, prefix)
    return f'(k={k}, j={j})'


def quality_cut(df, prefix, max_coeff_um):
    """Apply standard quality cuts: drop bad_fit and outliers."""
    n0 = len(df)
    bad_col = f'{prefix}_bad_fit'
    if bad_col in df.columns:
        df = df[~df[bad_col].astype(bool)].copy()
    elif 'bad_fit' in df.columns:
        df = df[~df['bad_fit'].astype(bool)].copy()
    cols = dz_coeff_columns(df, prefix)
    out_mask = df[cols].abs().gt(max_coeff_um).any(axis=1)
    df = df[~out_mask].copy()
    print(f'Quality cut: {len(df)}/{n0} rows kept '
          f'(prefix={prefix}, |c| < {max_coeff_um} \u00b5m)')
    return df


def selection_tag(chunk_names, param_set_filter):
    """Short slug describing the chunk selection, for the output label.

    Returns '' when both filters are None (i.e. use everything found).
    """
    if chunk_names:
        return '-'.join(chunk_names)
    if param_set_filter:
        # Normalize to a filesystem-safe slug
        return re.sub(r'[^A-Za-z0-9_]+', '_', param_set_filter).strip('_')
    return ''


<a id='data'></a>
## Data Access

In [ ]:
fit_files = find_fit_files(
    output_root, coord_sys,
    chunk_names=chunk_names, param_set_filter=param_set_filter)
print(f'Found {len(fit_files)} {coord_sys} fit files in {output_root}/')
if chunk_names:
    print(f'  chunk_names filter: {chunk_names}')
elif param_set_filter:
    print(f'  param_set_filter: {param_set_filter!r}')
for p in fit_files:
    print(f'  {p}')

df_all = load_all_fits(fit_files, source_label='chunk')
print(f'\nCombined: {len(df_all)} rows x {len(df_all.columns)} columns')
print(f'day_obs range: {df_all["day_obs"].min()} - {df_all["day_obs"].max()}')

In [ ]:
df = quality_cut(df_all, dz_prefix, max_coeff_um)
dz_cols = dz_coeff_columns(df, dz_prefix)
print(f'Number of DZ coefficient columns: {len(dz_cols)}')

# Unit normalization:
#   * alt is stored in RADIANS (from Butler meta) -> add alt_deg in degrees
#   * rotator_angle is already in DEGREES
if 'alt' in df.columns:
    df['alt_deg'] = np.rad2deg(df['alt'].astype(float))
    print(f'alt range (rad): {df["alt"].min():.3f} .. {df["alt"].max():.3f}')
    print(f'alt range (deg): {df["alt_deg"].min():.2f} .. {df["alt_deg"].max():.2f}')
if 'rotator_angle' in df.columns:
    print(f'rotator_angle range (deg): '
          f'{df["rotator_angle"].min():.2f} .. {df["rotator_angle"].max():.2f}')

# Sanity check on key metadata columns
for col in ['rotator_angle', 'alt', 'alt_deg', 'az'] + thermal_vars:
    n_valid = df[col].notna().sum() if col in df.columns else 0
    flag = 'OK' if col in df.columns else 'MISSING'
    print(f'  {col:<25s} {flag:<8s} valid={n_valid}/{len(df)}')
# ----- Resolve per-run output subdirectory now that data is loaded -----
_dmin = int(df['day_obs'].min())
_dmax = int(df['day_obs'].max())
if output_label is None:
    _sel = selection_tag(chunk_names, param_set_filter)
    if _sel:
        output_label = f'dz_analysis_{coord_sys}_{_sel}_{_dmin}_{_dmax}'
    else:
        output_label = f'dz_analysis_{coord_sys}_{_dmin}_{_dmax}'
output_dir = Path(output_root) / output_label
output_dir.mkdir(parents=True, exist_ok=True)

output_pdf_main          = str(output_dir / f'{output_label}.pdf')
output_pdf_thermal_grid  = str(output_dir / f'{output_label}_thermal_by_variable.pdf')
output_pdf_corr_groups   = str(output_dir / f'{output_label}_correlation_groups.pdf')
output_pdf_pairwise      = str(output_dir / f'{output_label}_pairwise.pdf')

print(f'\nOutput directory: {output_dir}')
for p in [output_pdf_main, output_pdf_thermal_grid,
          output_pdf_corr_groups, output_pdf_pairwise]:
    print(f'  {p}')

<a id='thermal-per-term'></a>
## 5. Thermal Correlations — per-DZ-term

For each selected DZ coefficient `(k, j)` produce a page of scatter plots
vs each thermal variable. Each point is one visit; a linear fit and Pearson
`r` are overlaid. Pages for this section are appended to the main PDF.

In [ ]:
def thermal_scatter_page(df, dz_col, k, j, thermal_vars):
    """Make a page of scatter plots: dz_col vs every thermal variable."""
    have = [tv for tv in thermal_vars if tv in df.columns]
    n = len(have)
    ncols = 4
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(14, 3.2 * nrows))
    fig.suptitle(
        f'DZ (k={k}, j={j}) = {dz_col}  vs thermal variables',
        fontsize=13, y=1.01)
    axes = np.atleast_1d(axes).ravel()
    for idx, tv in enumerate(have):
        ax = axes[idx]
        m = df[dz_col].notna() & df[tv].notna()
        x = df.loc[m, tv].values
        y = df.loc[m, dz_col].values
        ax.scatter(x, y, s=10, alpha=0.6, edgecolors='none')
        if len(x) > 2:
            c = np.polyfit(x, y, 1)
            r = np.corrcoef(x, y)[0, 1]
            xf = np.linspace(x.min(), x.max(), 50)
            ax.plot(xf, np.polyval(c, xf), 'r-', lw=1.2, alpha=0.8)
            ax.text(0.05, 0.92, f'r = {r:.3f}', transform=ax.transAxes,
                    fontsize=9, va='top',
                    bbox=dict(boxstyle='round,pad=0.2', fc='white', alpha=0.8))
        ax.set_xlabel(tv, fontsize=8)
        ax.set_ylabel(f'(k={k},j={j}) [µm]', fontsize=8)
        ax.tick_params(labelsize=7)
    for idx in range(n, len(axes)):
        axes[idx].set_visible(False)
    return fig

In [ ]:
thermal_figs = []
for k, j in dz_terms_for_thermal:
    col = f'{dz_prefix}_z{j}_c{k}'
    if col not in df.columns:
        print(f'Skipping {col}: not in table')
        continue
    if not any(tv in df.columns for tv in thermal_vars):
        print('No thermal variables present in tables — skipping section.')
        break
    fig = thermal_scatter_page(df, col, k, j, thermal_vars)
    thermal_figs.append(fig)
    plt.show()

<a id='thermal-by-var'></a>
## 6. Thermal Correlations — overview

Two views of DZ–thermal correlations, written together to
`{output_pdf_thermal_grid}`:

- **6.1 Per-variable grid** — for each thermal variable a page with a
  6×14 grid of DZ scatter plots (focal `k = 1..6`, pupil `j = 4..17`).
- **6.2 DZ × thermal correlation matrix** — single heatmap of Pearson `r`
  between every DZ coefficient and every thermal variable.

<a id='thermal-by-var-grid'></a>
### 6.1 Per-variable grid

For each thermal variable, show a 6×14 grid of DZ correlations: focal
Noll `k = 1..6` on rows, pupil Noll `j = 4..17` on columns. X axis is
the thermal variable for every panel.

In [ ]:
# Per-variable thermal correlation grids
def thermal_variable_grid_page(df, tv, k_range, j_range, dz_prefix):
    """One page: grid of DZ(k,j) vs single thermal variable tv."""
    k_list = list(k_range)
    j_list = list(j_range)
    nrows = len(k_list)
    ncols = len(j_list)
    fig, axes = plt.subplots(
        nrows, ncols,
        figsize=(1.6 * ncols, 1.5 * nrows + 0.8),
        sharex=True,
        layout='constrained')
    axes = np.atleast_2d(axes)
    fig.suptitle(
        f'DZ coefficients vs {tv}  ({coord_sys}, {dz_prefix})',
        fontsize=13)

    for ri, k in enumerate(k_list):
        for ci, j in enumerate(j_list):
            ax = axes[ri, ci]
            col = f'{dz_prefix}_z{j}_c{k}'
            if col not in df.columns:
                ax.set_visible(False)
                continue
            m = df[col].notna() & df[tv].notna()
            x = df.loc[m, tv].values
            y = df.loc[m, col].values
            if len(x) < 3:
                ax.set_visible(False)
                continue
            ax.scatter(x, y, s=4, alpha=0.5, edgecolors='none')
            c = np.polyfit(x, y, 1)
            xf = np.linspace(x.min(), x.max(), 30)
            ax.plot(xf, np.polyval(c, xf), 'r-', lw=1.0, alpha=0.8)
            r = np.corrcoef(x, y)[0, 1]
            ax.text(0.04, 0.93, f'r={r:+.2f}', transform=ax.transAxes,
                    fontsize=6, va='top',
                    bbox=dict(boxstyle='round,pad=0.1',
                              fc='white', alpha=0.7, lw=0))
            ax.axhline(0, color='gray', lw=0.3, alpha=0.5)
            ax.tick_params(labelsize=6)
            if ri == 0:
                ax.set_title(f'j={j} ({PUPIL_NAMES.get(j, f"Z{j}")})',
                             fontsize=7)
            if ci == 0:
                ax.set_ylabel(f'k={k}\n{FOCAL_NAMES.get(k, f"k{k}")}',
                              fontsize=7)
            if ri == nrows - 1:
                ax.set_xlabel(tv, fontsize=7)
    return fig


thermal_grid_figs = []
for tv in thermal_vars:
    if tv not in df.columns:
        print(f'Skipping {tv}: not in table')
        continue
    fig = thermal_variable_grid_page(
        df, tv, thermal_grid_k_range, thermal_grid_j_range, dz_prefix)
    thermal_grid_figs.append(fig)
    plt.show()
print(f'\nGenerated {len(thermal_grid_figs)} thermal-variable grid pages')

<a id='thermal-by-var-matrix'></a>
### 6.2 DZ × thermal correlation matrix

Heatmap of Pearson `r` between every DZ coefficient (rows, grouped by
pupil Noll `j`) and every thermal variable (columns). Useful for spotting
which aberration modes couple most strongly to which thermal inputs
before diving into per-variable grids.

In [ ]:
# DZ x thermal Pearson correlation matrix
# Rows: DZ coefficients (ordered by j, then k), Cols: thermal variables.
k_list = list(thermal_grid_k_range)
j_list = list(thermal_grid_j_range)
thermal_available = [tv for tv in thermal_vars if tv in df.columns]

dz_keys = [(j, k) for j in j_list for k in k_list]
dz_cols_grid = [f'{dz_prefix}_z{j}_c{k}' for (j, k) in dz_keys]
# Keep only columns actually in df
dz_cols_grid = [c for c in dz_cols_grid if c in df.columns]
dz_keys = [parse_jk(c, dz_prefix) for c in dz_cols_grid]

corr_mat = np.full((len(dz_cols_grid), len(thermal_available)), np.nan)
for i, col in enumerate(dz_cols_grid):
    for c_idx, tv in enumerate(thermal_available):
        m = df[col].notna() & df[tv].notna()
        if m.sum() > 2:
            corr_mat[i, c_idx] = np.corrcoef(
                df.loc[m, col].values, df.loc[m, tv].values)[0, 1]

# Row labels: "(k=K, j=J) Focal of Pupil"
row_labels = []
for (j, k) in dz_keys:
    fname = FOCAL_NAMES.get(k, f'k{k}')
    pname = PUPIL_NAMES.get(j, f'Z{j}')
    row_labels.append(f'k={k} j={j}  {fname} of {pname}')

# Separator indices between pupil-j groups (rows)
sep_rows = []
for i in range(1, len(dz_keys)):
    if dz_keys[i][0] != dz_keys[i - 1][0]:
        sep_rows.append(i - 0.5)

fig_dz_thermal_corr, ax = plt.subplots(
    figsize=(max(7, 0.5 * len(thermal_available) + 4),
             max(10, 0.18 * len(dz_cols_grid) + 1.2)),
    layout='constrained')
im = ax.imshow(corr_mat, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
ax.set_xticks(range(len(thermal_available)))
ax.set_xticklabels(thermal_available, rotation=70, fontsize=7,
                   ha='right')
ax.set_yticks(range(len(dz_cols_grid)))
ax.set_yticklabels(row_labels, fontsize=6)
for y in sep_rows:
    ax.axhline(y, color='black', lw=0.4, alpha=0.5)
ax.set_title(
    f'DZ × thermal Pearson correlation ({dz_prefix}, {coord_sys})\n'
    f'{len(dz_cols_grid)} DZ coefficients × {len(thermal_available)} '
    f'thermal variables',
    fontsize=11)
plt.colorbar(im, ax=ax, shrink=0.6, label='r')
plt.show()

# Quick print: top |r| per thermal variable
print('Top |r| per thermal variable:')
for c_idx, tv in enumerate(thermal_available):
    col = corr_mat[:, c_idx]
    if np.all(np.isnan(col)):
        continue
    i_best = int(np.nanargmax(np.abs(col)))
    print(f'  {tv:<25s}  r={col[i_best]:+.3f}  {row_labels[i_best]}')

<a id='corr-matrix-sec'></a>
## 7. DZ Correlation Matrix

Compute the pairwise Pearson correlation matrix over all DZ coefficients
and display it as a heatmap.

In [ ]:
corr_df = df[dz_cols].dropna()
print(f'Correlation sample size: {len(corr_df)} visits, {len(dz_cols)} DZ terms')
corr_matrix = corr_df.corr(method='pearson')

fig_corr, ax = plt.subplots(figsize=(14, 12))
im = ax.imshow(corr_matrix.values, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
labels = [dz_label(c, dz_prefix) for c in dz_cols]
ax.set_xticks(range(len(dz_cols)))
ax.set_xticklabels(labels, rotation=90, fontsize=5)
ax.set_yticks(range(len(dz_cols)))
ax.set_yticklabels(labels, fontsize=5)
ax.set_title(f'Pearson correlation — all {dz_prefix} DZ coefficients ({coord_sys})')
plt.colorbar(im, ax=ax, shrink=0.8, label='r')
plt.show()

<a id='corr-groups'></a>
## 8. Large DZ Correlations Grouped by Pupil Pair

Extract pairs with `|r| > corr_threshold`, group by pupil-Zernike pair
`(j_lo, j_hi)`, and display up to 8 scatter plots per page. Within each
group the plots are ordered by `(k_a, k_b)`, and the lower pupil Noll
index is always on the x-axis. All pages go to
`{output_pdf_corr_groups}`.

In [ ]:
# Extract all pairs with |r| > corr_threshold, grouped by pupil-Zernike pair
from collections import defaultdict

corr_threshold = 0.6

n = len(dz_cols)
iu, ju = np.triu_indices(n, k=1)
rs = corr_matrix.values[iu, ju]

# Filter by threshold
sig_mask = np.abs(rs) > corr_threshold
sig_indices = np.where(sig_mask)[0]
print(f'Pairs with |r| > {corr_threshold}: {len(sig_indices)} '
      f'(out of {len(rs)} total)')

# Group by sorted pupil-Zernike pair (j_lo, j_hi)
pupil_groups = defaultdict(list)
for idx in sig_indices:
    a = dz_cols[iu[idx]]
    b = dz_cols[ju[idx]]
    r = rs[idx]
    ja, ka = parse_jk(a, dz_prefix)
    jb, kb = parse_jk(b, dz_prefix)
    # Ensure col_a has the lower pupil index
    if ja > jb:
        a, b = b, a
        ja, ka, jb, kb = jb, kb, ja, ka
    pupil_groups[(ja, jb)].append((a, b, ka, kb, r))

# Sort within each group by (ka, kb)
for key in pupil_groups:
    pupil_groups[key].sort(key=lambda t: (t[2], t[3]))

# Print summary
print(f'\n{len(pupil_groups)} pupil-Zernike pair groups:')
for (ja, jb) in sorted(pupil_groups.keys()):
    pa = PUPIL_NAMES.get(ja, f'Z{ja}')
    pb = PUPIL_NAMES.get(jb, f'Z{jb}')
    entries = pupil_groups[(ja, jb)]
    print(f'  {pa} (Z{ja}) × {pb} (Z{jb}): {len(entries)} pairs')
    for a, b, ka, kb, r in entries:
        print(f'    {dz_label(a, dz_prefix)}  vs  {dz_label(b, dz_prefix)}  r={r:+.3f}')

In [ ]:
# Grouped scatter plots: one set of pages per pupil-Zernike pair.
# Use plain tight_layout with a reserved top rect so the suptitle never
# overlaps subplot titles (no colorbars on these pages).
corr_page_figs = []
plots_per_page = 8
ncols_per_page = 4

for (ja, jb) in sorted(pupil_groups.keys()):
    entries = pupil_groups[(ja, jb)]
    pa = PUPIL_NAMES.get(ja, f'Z{ja}')
    pb = PUPIL_NAMES.get(jb, f'Z{jb}')
    if ja == jb:
        page_title = f'{pa} (Z{ja}) self-correlations ({coord_sys})'
    else:
        page_title = f'{pa} (Z{ja}) × {pb} (Z{jb}) correlations ({coord_sys})'

    for page_start in range(0, len(entries), plots_per_page):
        page_entries = entries[page_start:page_start + plots_per_page]
        nrows_page = int(np.ceil(len(page_entries) / ncols_per_page))
        # Use layout=None so tight_layout(rect=...) works (reserves top margin)
        fig, axes = plt.subplots(
            nrows_page, ncols_per_page,
            figsize=(4.5 * ncols_per_page, 4.2 * nrows_page + 0.9),
            layout=None)
        axes = np.atleast_1d(axes).ravel()

        for idx, (a, b, ka, kb, r) in enumerate(page_entries):
            ax = axes[idx]
            m = df[a].notna() & df[b].notna()
            x = df.loc[m, a].values
            y = df.loc[m, b].values
            ax.scatter(x, y, s=12, alpha=0.6, edgecolors='none')
            if len(x) > 2:
                c = np.polyfit(x, y, 1)
                xf = np.linspace(x.min(), x.max(), 50)
                ax.plot(xf, np.polyval(c, xf), 'r-', lw=1.2, alpha=0.8)
            ax.axhline(0, color='gray', lw=0.5, alpha=0.5)
            ax.axvline(0, color='gray', lw=0.5, alpha=0.5)
            ax.set_xlabel(dz_label(a, dz_prefix), fontsize=8)
            ax.set_ylabel(dz_label(b, dz_prefix), fontsize=8)
            ax.set_title(f'r = {r:+.3f}   (n={m.sum()})', fontsize=10)

        for idx in range(len(page_entries), len(axes)):
            axes[idx].set_visible(False)

        page_num = page_start // plots_per_page + 1
        total_pages = int(np.ceil(len(entries) / plots_per_page))
        suffix = f' ({page_num}/{total_pages})' if total_pages > 1 else ''
        fig.suptitle(page_title + suffix, fontsize=13)
        # Reserve top 8% of figure for suptitle so it never overlaps row 1.
        fig.tight_layout(rect=[0, 0, 1, 0.92])
        corr_page_figs.append(fig)
        plt.show()

print(f'\nTotal correlation scatter pages: {len(corr_page_figs)}')

<a id='corr-expected'></a>
### 9. Expected Astigmatism-Symmetry Correlations

Scatter plots of DZ coefficient pairs expected to correlate by the
astigmatism symmetry between focal Noll k = 5 ↔ 6 and pupil Noll
j = 5 ↔ 6 (`expected_astig_pairs`). This page is appended to the main PDF.

In [ ]:
ncols = 3
nrows = int(np.ceil(len(expected_astig_pairs) / ncols))
fig_expected, axes = plt.subplots(nrows, ncols, figsize=(4.5 * ncols, 4.0 * nrows))
axes = np.atleast_1d(axes).ravel()

for idx, ((ka, ja), (kb, jb)) in enumerate(expected_astig_pairs):
    a = f'{dz_prefix}_z{ja}_c{ka}'
    b = f'{dz_prefix}_z{jb}_c{kb}'
    ax = axes[idx]
    if a not in df.columns or b not in df.columns:
        ax.set_visible(False)
        print(f'Skipping ({ka},{ja}) vs ({kb},{jb}): column missing')
        continue
    m = df[a].notna() & df[b].notna()
    x = df.loc[m, a].values
    y = df.loc[m, b].values
    r = np.corrcoef(x, y)[0, 1] if len(x) > 2 else np.nan
    ax.scatter(x, y, s=12, alpha=0.6, edgecolors='none')
    if len(x) > 2:
        c = np.polyfit(x, y, 1)
        xf = np.linspace(x.min(), x.max(), 50)
        ax.plot(xf, np.polyval(c, xf), 'r-', lw=1.2, alpha=0.8)
    ax.axhline(0, color='gray', lw=0.5, alpha=0.5)
    ax.axvline(0, color='gray', lw=0.5, alpha=0.5)
    ax.set_xlabel(f'(k={ka}, j={ja})  [µm]', fontsize=10)
    ax.set_ylabel(f'(k={kb}, j={jb})  [µm]', fontsize=10)
    ax.set_title(f'r = {r:+.3f}   (n={m.sum()})', fontsize=11)

for idx in range(len(expected_astig_pairs), len(axes)):
    axes[idx].set_visible(False)

fig_expected.suptitle(
    f'Expected astigmatism-symmetry DZ correlations ({dz_prefix}, {coord_sys})',
    fontsize=13, y=1.005)
plt.show()

<a id='pairwise'></a>
## 10. Pairwise (k1,j1) vs (k2,j2) scan

For every combination of pupil Noll indices `j1, j2 = 4..14` produce a
page with 36 scatter plots covering the 6×6 grid of focal Noll indices
`k1, k2 = 1..6`. Each subplot shows `(k2,j2)` (y-axis) vs `(k1,j1)`
(x-axis), titled `(k2,j2) vs (k1,j1)`. Total: 11 × 11 = 121 pages,
written to `{output_pdf_pairwise}`.

In [ ]:
# Pairwise (k1,j1) vs (k2,j2) scan — one page per (j1, j2) combination.
# Each page has a 6x6 grid for k1, k2 = 1..6.
# With 121 pages, we write directly to the PDF here rather than keeping
# all figures in memory.
def pairwise_page(df, j1, j2, k_range, dz_prefix):
    k_list = list(k_range)
    n = len(k_list)
    fig, axes = plt.subplots(
        n, n,
        figsize=(2.0 * n, 2.0 * n + 0.8),
        layout='constrained')
    axes = np.atleast_2d(axes)
    p1 = PUPIL_NAMES.get(j1, f'Z{j1}')
    p2 = PUPIL_NAMES.get(j2, f'Z{j2}')
    fig.suptitle(
        f'({p2} Z{j2}) vs ({p1} Z{j1})  —  j2={j2} vs j1={j1}  ({coord_sys})',
        fontsize=12)

    for ri, k2 in enumerate(k_list):
        for ci, k1 in enumerate(k_list):
            ax = axes[ri, ci]
            col_x = f'{dz_prefix}_z{j1}_c{k1}'
            col_y = f'{dz_prefix}_z{j2}_c{k2}'
            if col_x not in df.columns or col_y not in df.columns:
                ax.set_visible(False)
                continue
            # Skip identical-column self-correlation when (k1,j1)==(k2,j2)
            if col_x == col_y:
                ax.set_visible(False)
                continue
            m = df[col_x].notna() & df[col_y].notna()
            if m.sum() < 3:
                ax.set_visible(False)
                continue
            x = df.loc[m, col_x].values
            y = df.loc[m, col_y].values
            ax.scatter(x, y, s=4, alpha=0.5, edgecolors='none')
            c = np.polyfit(x, y, 1)
            xf = np.linspace(x.min(), x.max(), 30)
            ax.plot(xf, np.polyval(c, xf), 'r-', lw=1.0, alpha=0.8)
            r = np.corrcoef(x, y)[0, 1]
            ax.axhline(0, color='gray', lw=0.3, alpha=0.5)
            ax.axvline(0, color='gray', lw=0.3, alpha=0.5)
            ax.set_title(
                f'(k={k2},j={j2}) vs (k={k1},j={j1})  r={r:+.2f}',
                fontsize=7)
            ax.tick_params(labelsize=6)
            if ri == n - 1:
                ax.set_xlabel(f'k1={k1} ({FOCAL_NAMES.get(k1,"")})',
                              fontsize=7)
            if ci == 0:
                ax.set_ylabel(f'k2={k2} ({FOCAL_NAMES.get(k2,"")})',
                              fontsize=7)
    return fig


# Write all pages directly to the pairwise PDF
Path(output_pdf_pairwise).parent.mkdir(parents=True, exist_ok=True)
j_list = list(pairwise_j_range)
n_total = len(j_list) ** 2
print(f'Writing {n_total} pairwise pages to {output_pdf_pairwise}...')

with PdfPages(output_pdf_pairwise) as pdf:
    page_ct = 0
    for j1 in j_list:
        for j2 in j_list:
            fig = pairwise_page(df, j1, j2, pairwise_k_range, dz_prefix)
            pdf.savefig(fig, bbox_inches='tight')
            plt.close(fig)
            page_ct += 1
            if page_ct % 20 == 0:
                print(f'  ...wrote {page_ct}/{n_total}')
print(f'Done: {page_ct} pages -> {output_pdf_pairwise}')

<a id='lut'></a>
## 11. LUT: DZ vs Rotator and Elevation

Inputs to a future Look-Up Table for AOS feed-forward correction.

* **Rotator scan** — plot every DZ coefficient vs `rotator_angle` for visits
  with `alt` within ±3 deg of 70 deg.
* **Elevation scan** — plot every DZ coefficient vs `alt` for visits with
  `rotator_angle` within ±3 deg of 0 deg.

In [ ]:
def grid_scan_page(df, x_col, dz_cols, prefix, title, x_label, x_unit='deg'):
    """Plot every DZ coefficient vs x_col in a tight grid."""
    n = len(dz_cols)
    ncols = 6
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols,
                             figsize=(2.3 * ncols, 1.9 * nrows),
                             sharex=True)
    axes = np.atleast_1d(axes).ravel()
    for idx, col in enumerate(dz_cols):
        ax = axes[idx]
        m = df[col].notna() & df[x_col].notna()
        x = df.loc[m, x_col].values
        y = df.loc[m, col].values
        ax.scatter(x, y, s=6, alpha=0.6, edgecolors='none')
        ax.axhline(0, color='gray', lw=0.5, alpha=0.6)
        ax.set_title(short_label(col, prefix) if prefix in col
                     else col, fontsize=7)
        ax.tick_params(labelsize=6)
    for idx in range(n, len(axes)):
        axes[idx].set_visible(False)
    fig.suptitle(title, fontsize=12, y=1.005)
    fig.supxlabel(f'{x_label} [{x_unit}]', fontsize=10)
    fig.supylabel('DZ coefficient [µm]', fontsize=10)
    return fig

In [ ]:
# DZ vs rotator angle, at fixed elevation band (all in degrees)
elev_lo = elev_center_for_rotator_scan - elev_tol
elev_hi = elev_center_for_rotator_scan + elev_tol
mask_elev = df['alt_deg'].between(elev_lo, elev_hi)
df_rot = df[mask_elev & df['rotator_angle'].notna()].copy()
print(f'Elevation slice {elev_lo:.0f}–{elev_hi:.0f} deg: '
      f'{len(df_rot)}/{len(df)} visits')

if len(df_rot) >= 5:
    fig_rot = grid_scan_page(
        df_rot, 'rotator_angle', dz_cols, dz_prefix,
        title=(f'DZ vs rotator angle  (elevation = '
               f'{elev_center_for_rotator_scan:.0f} ± {elev_tol:.0f} deg, '
               f'n={len(df_rot)}, {coord_sys})'),
        x_label='rotator angle')
    plt.show()
else:
    fig_rot = None
    print('Too few visits in elevation slice for plot.')

In [ ]:
# DZ vs elevation (degrees), at fixed rotator-angle band
rot_lo = rot_center_for_elev_scan - rot_tol
rot_hi = rot_center_for_elev_scan + rot_tol
mask_rot = df['rotator_angle'].between(rot_lo, rot_hi)
df_elev = df[mask_rot & df['alt_deg'].notna()].copy()
print(f'Rotator slice {rot_lo:.0f}–{rot_hi:.0f} deg: '
      f'{len(df_elev)}/{len(df)} visits')

if len(df_elev) >= 5:
    fig_elev = grid_scan_page(
        df_elev, 'alt_deg', dz_cols, dz_prefix,
        title=(f'DZ vs elevation  (rotator = '
               f'{rot_center_for_elev_scan:.0f} ± {rot_tol:.0f} deg, '
               f'n={len(df_elev)}, {coord_sys})'),
        x_label='elevation')
    plt.show()
else:
    fig_elev = None
    print('Too few visits in rotator slice for plot.')

<a id='radial'></a>
## 12. Cylindrical (Radial) Pupil Zernike Combinations

The Noll-indexed Zernikes pair azimuthal cos/sin terms. The amplitude
$Z_m^{(r)} = \sqrt{Z_\text{cos}^2 + Z_\text{sin}^2}$ is rotationally invariant
and carries the image-quality information; the associated angle
$\arctan(Z_\text{sin}/Z_\text{cos})$ is less interesting for AOS diagnostics.

Purely radial terms (m = 0) are left unchanged. Combining only the **pupil**
Zernikes (keeping focal-plane Zernikes separate, since any camera-side optical
effect may add rotator-angle dependence beyond the OCS→CCS transform), the 21
pupil Noll indices Z4–Z24 reduce to **12 radial terms**:

| Short name        | Description           | Noll indices combined |
|-------------------|-----------------------|-----------------------|
| `Z4`              | defocus               | 4                     |
| `Z_astig`         | astigmatism           | 5, 6                  |
| `Z_coma`          | coma                  | 7, 8                  |
| `Z_trefoil`       | trefoil               | 9, 10                 |
| `Z11`             | spherical             | 11                    |
| `Z_2ndastig`      | 2nd astigmatism       | 12, 13                |
| `Z_tetrafoil`     | tetrafoil             | 14, 15                |
| `Z_2ndcoma`       | 2nd coma              | 16, 17                |
| `Z_2ndtrefoil`    | 2nd trefoil           | 18, 19                |
| `Z_pentafoil`     | pentafoil             | 20, 21                |
| `Z22`             | 2nd spherical         | 22                    |
| `Z_3rdastig`      | 3rd astigmatism       | 23, 24                |

For a DZ coefficient the combination is applied per focal-plane Noll index `k`:
$c_k^{(\text{radial})} = \sqrt{c_k^{(\text{cos})2} + c_k^{(\text{sin})2}}$.

In [ ]:
# Radial pupil Zernike definitions (Noll convention)
# Each entry: (short_name, [list of Noll indices to combine])
RADIAL_PUPIL_DEFS = [
    ('Z4',            [4]),
    ('Z_astig',       [5, 6]),
    ('Z_coma',        [7, 8]),
    ('Z_trefoil',     [9, 10]),
    ('Z11',           [11]),
    ('Z_2ndastig',    [12, 13]),
    ('Z_tetrafoil',   [14, 15]),
    ('Z_2ndcoma',     [16, 17]),
    ('Z_2ndtrefoil',  [18, 19]),
    ('Z_pentafoil',   [20, 21]),
    ('Z22',           [22]),
    ('Z_3rdastig',    [23, 24]),
]


def radial_combine(values_list):
    """Return sqrt(sum(v**2)) with NaN propagation element-wise."""
    arr = np.asarray(values_list, dtype=float)
    if arr.ndim == 1:
        return float(np.sqrt(np.nansum(arr**2))) if not np.any(np.isnan(arr)) else np.nan
    # axis=0: combine across the Noll-index axis
    nan_mask = np.isnan(arr).any(axis=0)
    out = np.sqrt(np.nansum(arr**2, axis=0))
    out[nan_mask] = np.nan
    return out


def add_radial_dz_columns(df, prefix, radial_defs, max_k=6):
    """Add radial-combined DZ columns to df.

    Column name: '{prefix}_{short_name}_c{k}' for k = 1..max_k.
    Only adds a column if ALL required source columns are present.

    Returns list of newly-added column names.
    """
    added = []
    for short_name, nolls in radial_defs:
        for k in range(1, max_k + 1):
            src_cols = [f'{prefix}_z{j}_c{k}' for j in nolls]
            missing = [c for c in src_cols if c not in df.columns]
            if missing:
                continue
            new_col = f'{prefix}_{short_name}_c{k}'
            vals = np.stack([df[c].values.astype(float) for c in src_cols])
            df[new_col] = radial_combine(vals)
            added.append(new_col)
    return added


# Determine max focal k from prefix
max_focal_k = 6 if dz_prefix == 'z1toz6' else 3
radial_cols = add_radial_dz_columns(df, dz_prefix, RADIAL_PUPIL_DEFS,
                                    max_k=max_focal_k)
print(f'Added {len(radial_cols)} radial DZ columns '
      f'({len(RADIAL_PUPIL_DEFS)} pupil terms × {max_focal_k} focal k)')
print('Sample:', radial_cols[:6], '...')

<a id='radial-corr'></a>
### 12.1 Radial DZ Correlation Matrix

Pearson-correlation heatmap using the 12 radial pupil combinations with all
focal-plane Noll indices retained.

In [ ]:
def radial_label(col, prefix):
    """Label like 'Z_astig (k=3)' from 'z1toz6_Z_astig_c3'."""
    m = re.match(rf'^{re.escape(prefix)}_(.+)_c(\d+)$', col)
    if not m:
        return col
    return f'{m.group(1)} (k={m.group(2)})'


radial_corr_df = df[radial_cols].dropna()
print(f'Radial correlation sample size: {len(radial_corr_df)} visits, '
      f'{len(radial_cols)} radial DZ terms')
radial_corr_matrix = radial_corr_df.corr(method='pearson')

fig_rcorr, ax = plt.subplots(figsize=(12, 11))
im = ax.imshow(radial_corr_matrix.values, cmap='RdBu_r',
               vmin=-1, vmax=1, aspect='auto')
rlabels = [radial_label(c, dz_prefix) for c in radial_cols]
ax.set_xticks(range(len(radial_cols)))
ax.set_xticklabels(rlabels, rotation=90, fontsize=6)
ax.set_yticks(range(len(radial_cols)))
ax.set_yticklabels(rlabels, fontsize=6)
ax.set_title(
    f'Pearson correlation — radial-combined pupil DZ terms ({coord_sys})')
plt.colorbar(im, ax=ax, shrink=0.8, label='r')
plt.show()

<a id='radial-trio'></a>
### 12.2 Radial-Combination Trio Plots

For each radial pupil term, construct focal-plane Trio maps:

* **Data** — measured per-donut zk minus the fitted linear (focal-plane k=1,2,3)
  component, combined radially across the Noll pair.
* **Intrinsic** — the batoid intrinsic model value, radially combined.
* **Data – Intrinsic** — residual.

Reads the HDF5 `donuts` tables for every chunk and (re)constructs the
per-donut fitted zk from the matching `_fits.parquet` (subtracting only
focal-plane Noll k = 1, 2, 3 — the `z1toz3` fit).

**Color scale:** the Data 3–97 percentile range is used for all three panels
so Data / Model / Residual are directly comparable.

In [ ]:
from astropy.table import QTable, vstack
from scipy.stats import binned_statistic_2d

# Pull helpers from our code/ package
sys.path.insert(0, 'code')
from dz_plotting import reconstruct_zk_fit
from dz_fitting import derive_noll_indices


def hdf5_for_fitfile(fit_path, coord_sys):
    """Derive the HDF5 path corresponding to a fit parquet."""
    p = Path(fit_path)
    stem = p.stem
    if coord_sys == 'CCS' and stem.endswith('_ccs_fits'):
        base = stem[:-len('_ccs_fits')]
    elif stem.endswith('_fits'):
        base = stem[:-len('_fits')]
    else:
        base = stem
    return p.parent / f'{base}.hdf5'


def load_donuts_for_trio(fit_files, coord_sys, fit_prefix='z1toz3'):
    """Load 'donuts' tables for all chunks and attach reconstructed zk_fit.

    Returns (aosTable, iZs, iZidx) where iZs is the Noll-index list.
    """
    tables = []
    iZs_ref = None
    for fit_path in fit_files:
        h5 = hdf5_for_fitfile(fit_path, coord_sys)
        if not h5.exists():
            print(f'  [skip] {h5.name} not found')
            continue
        aos = QTable.read(str(h5), path='donuts')
        visit_info = QTable.read(str(h5), path='visits')

        # Noll indices
        nZk = np.stack(aos[f'zk_{coord_sys}']).shape[1]
        noll_arr = (np.array(visit_info['nollIndices'][0])
                    if 'nollIndices' in visit_info.colnames else None)
        iZs, iZidx = derive_noll_indices(nZk, noll_arr)
        if iZs_ref is None:
            iZs_ref, iZidx_ref = iZs, iZidx
        elif iZs != iZs_ref:
            print(f'  [warn] {h5.name}: iZs {iZs} != {iZs_ref}; skipping')
            continue

        # Fit table and reconstruct zk_fit_{fit_prefix}
        ft = QTable.read(str(fit_path), format='parquet')
        max_k = 3 if fit_prefix == 'z1toz3' else 6
        reconstruct_zk_fit(aos, ft, coord_sys, iZs,
                           prefix=fit_prefix, max_focal_noll=max_k)
        tables.append(aos)
        print(f'  {h5.name}: {len(aos)} donuts')

    if not tables:
        raise FileNotFoundError('No HDF5 donut tables found')
    combined = vstack(tables, metadata_conflicts='silent')
    print(f'\nCombined donuts: {len(combined)} rows')
    return combined, iZs_ref, iZidx_ref

In [ ]:
# Load donut-level data for all chunks.  Use the z1toz3 fit so that only
# focal-plane k = 1, 2, 3 components are subtracted from the data.
aosTable_all, iZs_all, iZidx_all = load_donuts_for_trio(
    fit_files, coord_sys, fit_prefix='z1toz3')

In [ ]:
def plot_radial_trio(aosTable, short_name, nolls, iZidx, coord_sys,
                     fit_prefix='z1toz3', statistic='median',
                     plo=5.0, phi=95.0):
    """Trio plot for a radial-combined pupil Zernike term.

    Each of the three panels bins per-donut values on the focal plane:
      - Data: sqrt(sum((zk - zk_fit)^2)) over Noll indices in `nolls`.
      - Intrinsic: sqrt(sum(zk_intrinsic^2)).
      - Data - Intrinsic: signed difference (per-donut, then combined).

    Color scale: the Data 5-95 pctile range is used as vmin/vmax for all
    three panels (so Model and Residual are directly comparable to Data).
    """
    nsteps = 18 * 4 + 1
    fpradius = 1.8
    xbins = np.linspace(-fpradius, fpradius, nsteps)
    ybins = np.linspace(-fpradius, fpradius, nsteps)

    xval = np.rad2deg(np.array(aosTable[f'thy_{coord_sys}_extra']))
    yval = np.rad2deg(np.array(aosTable[f'thx_{coord_sys}_extra']))

    zk_data_all = np.stack(aosTable[f'zk_{coord_sys}'])
    zk_fit_all = np.stack(aosTable[f'zk_fit_{fit_prefix}'])
    zk_model_all = np.stack(aosTable[f'zk_intrinsic_{coord_sys}'])

    data_stack = []
    model_stack = []
    for j in nolls:
        if j not in iZidx:
            return None
        idx = iZidx[j]
        data_stack.append(zk_data_all[:, idx] - zk_fit_all[:, idx])
        model_stack.append(zk_model_all[:, idx])
    data_stack = np.stack(data_stack)
    model_stack = np.stack(model_stack)

    zval_data = np.sqrt(np.nansum(data_stack ** 2, axis=0))
    zval_model = np.sqrt(np.nansum(model_stack ** 2, axis=0))
    zval_residual = zval_data - zval_model

    # Color scale from Data percentiles; applied to all three panels.
    vmin, vmax = np.nanpercentile(zval_data, [plo, phi])

    fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))
    for ax, zval, cmap, title_str in [
        (axes[0], zval_data, 'viridis',
         f'{short_name} Data (linear fit subtracted)'),
        (axes[1], zval_model, 'viridis', f'{short_name} Intrinsic Model'),
        (axes[2], zval_residual, 'RdBu_r', f'{short_name} Data − Model'),
    ]:
        stat_val, _, _, _ = binned_statistic_2d(
            xval, yval, zval, statistic=statistic, bins=[xbins, ybins])
        im = ax.imshow(stat_val.T, origin='lower',
                       extent=[xbins[0], xbins[-1], ybins[0], ybins[-1]],
                       cmap=cmap, interpolation='none', aspect='equal',
                       vmin=vmin, vmax=vmax)
        plt.colorbar(im, ax=ax, shrink=0.8, label='µm')
        ax.set_xlabel(f'thy_{coord_sys} [deg]')
        ax.set_ylabel(f'thx_{coord_sys} [deg]')
        ax.set_title(title_str)
        ax.set_aspect('equal')

    noll_str = ','.join(str(j) for j in nolls)
    fig.suptitle(
        f'Radial {short_name}  (Noll {noll_str}, {coord_sys}, {statistic})  '
        f'color: Data {plo:.0f}-{phi:.0f}% = [{vmin:.3f}, {vmax:.3f}] µm',
        fontsize=13, y=1.02)
    print(f'{short_name}: Data σ={np.nanstd(zval_data):.3f}, '
          f'Model σ={np.nanstd(zval_model):.3f}, '
          f'Resid σ={np.nanstd(zval_residual):.3f} µm  '
          f'color range [{vmin:.3f}, {vmax:.3f}]')
    return fig

In [ ]:
trio_figs = []
for short_name, nolls in RADIAL_PUPIL_DEFS:
    if not all(j in iZidx_all for j in nolls):
        print(f'{short_name}: skip (Noll {nolls} not all present)')
        continue
    fig = plot_radial_trio(aosTable_all, short_name, nolls,
                           iZidx_all, coord_sys,
                           fit_prefix='z1toz3', statistic='median')
    if fig is not None:
        trio_figs.append(fig)
        plt.show()

<a id='output'></a>
## 13. Output PDFs

The analysis produces four PDFs:

* `dz_analysis_{coord_sys}.pdf` — main PDF (per-DZ-term thermals, correlation heatmap,
  expected-astig symmetry, LUT scans, radial heatmap + trio plots).
* `dz_analysis_{coord_sys}_thermal_by_variable.pdf` — one page per thermal variable
  with a 6×11 grid of DZ correlations.
* `dz_analysis_{coord_sys}_correlation_groups.pdf` — grouped-by-pupil-pair
  scatter pages for `|r| > corr_threshold`.
* `dz_analysis_{coord_sys}_pairwise.pdf` — 121 pages of (k1,j1) vs (k2,j2)
  scans (written directly in section 10).

In [ ]:
Path(output_pdf_main).parent.mkdir(parents=True, exist_ok=True)

# Main PDF: per-DZ-term thermal scatter, correlation heatmap, expected-astig,
# LUT scans, radial heatmap, radial trio plots
with PdfPages(output_pdf_main) as pdf:
    for fig in thermal_figs:
        pdf.savefig(fig, bbox_inches='tight')
    pdf.savefig(fig_corr, bbox_inches='tight')
    pdf.savefig(fig_expected, bbox_inches='tight')
    if fig_rot is not None:
        pdf.savefig(fig_rot, bbox_inches='tight')
    if fig_elev is not None:
        pdf.savefig(fig_elev, bbox_inches='tight')
    pdf.savefig(fig_rcorr, bbox_inches='tight')
    for fig in trio_figs:
        pdf.savefig(fig, bbox_inches='tight')
print(f'Wrote main PDF: {output_pdf_main}')

# Thermal: per-variable grids + DZ×thermal correlation matrix
with PdfPages(output_pdf_thermal_grid) as pdf:
    for fig in thermal_grid_figs:
        pdf.savefig(fig, bbox_inches='tight')
    pdf.savefig(fig_dz_thermal_corr, bbox_inches='tight')
print(f'Wrote thermal-by-variable PDF: {output_pdf_thermal_grid}')

# Grouped |r|>threshold scatter pages -> separate PDF
with PdfPages(output_pdf_corr_groups) as pdf:
    for fig in corr_page_figs:
        pdf.savefig(fig, bbox_inches='tight')
print(f'Wrote correlation-groups PDF: {output_pdf_corr_groups}')

# (Pairwise PDF was written directly by the pairwise cell.)
print(f'Pairwise PDF already written to: {output_pdf_pairwise}')